In [ ]:
import glob
import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ee
import geemap
import datetime as dt
from sklearn.cluster import DBSCAN

warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
MODELS_BASE_PATH = './results/models/'
RASTER_INPUT_DIR = './raster_test/'
OUT_DIR = './test_results'
BAND_NAMES_JSON = 'band_names.json'
EE_PROJECT = 'ee-mateusfulan-research'
THRESHOLD = 0.5 

QC_PATTERNS = [
    'MOD13Q1_SummaryQA_0_1-', 'MCD15A3H_FparLai_QC_bit0-',
    'MCD15A3H_FparLai_QC_3_4-', 'MOD21A1D_QC_0_1-',
    'MOD21A1D_QC_4_5-', 'MCD18A1_DSR_Quality_0_1-',
]
PERIODS = ['5B', '4B', '3B', '2B', '1B', '0B']
REDUX_COLS = ['land_cover_class', 'elevation']

os.makedirs(OUT_DIR, exist_ok=True)
ee.Initialize(project=EE_PROJECT)

def preprocess_dataframe(df, variant):
    """Replicates training preprocessing: QC masking and feature selection."""
    df_proc = df.copy()
    
    for p in PERIODS:
        # MODIS NDVI/EVI QC
        qa = f'MOD13Q1_SummaryQA_0_1-{p}'
        if qa in df_proc.columns:
            bad = df_proc[qa] != 0
            for v in ['NDVI', 'EVI']:
                col = f'{v}_{p}'
                if col in df_proc.columns:
                    df_proc.loc[bad, col] = np.nan

        # LAI/FPAR QC
        qa0 = f'MCD15A3H_FparLai_QC_bit0-{p}'
        qa1 = f'MCD15A3H_FparLai_QC_3_4-{p}'
        if qa0 in df_proc.columns and qa1 in df_proc.columns:
            bad = (df_proc[qa0] != 0) | (df_proc[qa1] != 0)
            for v in ['LAI', 'FPAR']:
                col = f'{v}_{p}'
                if col in df_proc.columns:
                    df_proc.loc[bad, col] = np.nan

    # Drop non-feature columns
    qa_cols = [c for c in df_proc.columns if any(c.startswith(pat) for pat in QC_PATTERNS)]
    lst_cols = [c for c in df_proc.columns if c.startswith('LST_')]
    df_proc.drop(columns=qa_cols + lst_cols, inplace=True)

    if variant == 'reduced':
        df_proc.drop(columns=[c for c in REDUX_COLS if c in df_proc.columns], inplace=True)
    
    return df_proc

def get_ee_validation_data(date_str, dataset_type, threshold, region):
    """Retrieves ground truth (FIRMS/VIIRS) from Earth Engine for comparison."""
    start_dt = dt.datetime.strptime(date_str, '%Y%m%d')
    end_dt = start_dt + dt.timedelta(days=1)
    
    collection = 'FIRMS' if 'FIRMS' in dataset_type else 'ID_OF_VIIRS_COLLECTION' # Update as needed
    
    fire_mask = (ee.ImageCollection(collection)
                 .select('confidence')
                 .filterDate(start_dt.strftime('%Y-%m-%d'), end_dt.strftime('%Y-%m-%d'))
                 .map(lambda i: i.clip(region))
                 .max())
    
    # Scale: MODIS ~1000m, VIIRS ~375m
    scale = 1000 if 'FIRMS' in dataset_type else 375
    
    vectors = fire_mask.updateMask(fire_mask.gte(threshold)).sample(
        region=region.geometry(),
        scale=scale,
        numPixels=1e13,
        geometries=True
    )
    return vectors

def run_prediction_pipeline():
    """Main execution loop for all model combinations."""
    
    combinations = []
    for dataset in ['FIRMS', 'FIRMS$CHIRPS']:
        thresholds = ['80', '90', '95']
        for thresh in thresholds:
            for m_type in ['full', 'reduced']:
                combinations.append({
                    'dataset': dataset.replace('$', '-'),
                    'variant': m_type,
                    'threshold': f"{thresh}_conf",
                    'raw_thresh': thresh
                })

    with open(BAND_NAMES_JSON) as f:
        band_names = json.load(f)

    for combo in combinations:
        model_filename = f"RF_{combo['dataset']}_{combo['threshold']}_{combo['variant']}.joblib"
        model_path = os.path.join(MODELS_BASE_PATH, model_filename)
        
        if not os.path.exists(model_path):
            continue

        model = joblib.load(model_path)
        raster_pattern = f"COMBUSTION_{combo['dataset'].replace('-', '_')}_*_stack.tif"
        raster_list = glob.glob(os.path.join(RASTER_INPUT_DIR, raster_pattern))

        for raster_path in raster_list:
            date_str = os.path.basename(raster_path).split('_')[-2]
            
            with rasterio.open(raster_path) as src:
                profile = src.profile
                data = src.read().astype(np.float32)
                transform = src.transform
                nodata = src.nodata
                
            # Flatten for Inference
            n_bands, height, width = data.shape
            flat_data = data.reshape(n_bands, -1).T
            df = pd.DataFrame(flat_data, columns=band_names)
            if nodata is not None:
                df.replace(nodata, np.nan, inplace=True)

            valid_mask = df.notna().any(axis=1).values
            df_proc = preprocess_dataframe(df, combo['variant'])
            
            # Align features with model
            if hasattr(model, 'feature_names_in_'):
                df_proc = df_proc.reindex(columns=list(model.feature_names_in_))

            # Predict
            prob_flat = np.full(height * width, np.nan, dtype=np.float32)
            if valid_mask.any():
                prob_flat[valid_mask] = model.predict_proba(df_proc[valid_mask])[:, 1]
            
            prob_map = prob_flat.reshape(height, width)

            # --- Save Outputs ---
            out_name = f"fire_prob_{combo['dataset']}_{combo['variant']}_{date_str}.tif"
            profile.update(count=1, dtype='float32', nodata=-9999, compress='lzw')
            
            with rasterio.open(os.path.join(OUT_DIR, out_name), 'w', **profile) as dst:
                dst.write(np.where(np.isnan(prob_map), -9999, prob_map).astype(np.float32), 1)

if __name__ == "__main__":
    run_prediction_pipeline()